In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib
from pathlib import Path
import glob

# ======= Configuration =======
# Ensure these match your training constants exactly
TIME_COL   = "TIMESTAMP"
TARGET_COL = "TARGETVAR"
BASE_FEATS = ["U10", "V10", "U100", "V100"]
LAGS_Y     = [1, 2, 3, 6, 12, 24]  # Updated to match new training code
LAGS_SPEED = [1, 3, 6]
ROLLS_Y    = [6, 12, 24]
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======= 1. Feature Engineering (Must match Training EXACTLY) =======
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # wind speed & direction
    out["speed10"]  = np.sqrt(out["U10"]**2  + out["V10"]**2)
    out["speed100"] = np.sqrt(out["U100"]**2 + out["V100"]**2)

    # === NEW: Physics feature ===
    out["speed100_cubed"] = out["speed100"] ** 3

    dir10  = np.arctan2(out["V10"],  out["U10"])
    dir100 = np.arctan2(out["V100"], out["U100"])
    out["dir10_sin"], out["dir10_cos"]   = np.sin(dir10),  np.cos(dir10)
    out["dir100_sin"], out["dir100_cos"] = np.sin(dir100), np.cos(dir100)

    # shear & veer
    out["shear_speed"] = out["speed100"] - out["speed10"]
    veer = dir100 - dir10
    out["veer_sin"], out["veer_cos"] = np.sin(veer), np.cos(veer)

    # time features
    out["hour"] = pd.to_datetime(out[TIME_COL]).dt.hour
    out["day"]  = pd.to_datetime(out[TIME_COL]).dt.dayofyear
    out["hour_sin"] = np.sin(2*np.pi*out["hour"]/24.0)
    out["hour_cos"] = np.cos(2*np.pi*out["hour"]/24.0)
    out["day_sin"]  = np.sin(2*np.pi*out["day"]/366.0)
    out["day_cos"]  = np.cos(2*np.pi*out["day"]/366.0)

    # target lags
    for L in LAGS_Y:
        out[f"y_lag{L}"] = out[TARGET_COL].shift(L)

    # rolling means
    for W in ROLLS_Y:
        out[f"y_roll{W}"] = out[TARGET_COL].shift(1).rolling(W, min_periods=W).mean()

    # speed lags
    for L in LAGS_SPEED:
        out[f"speed10_lag{L}"]  = out["speed10"].shift(L)
        out[f"speed100_lag{L}"] = out["speed100"].shift(L)

    return out

def build_feat_list():
    return (
        BASE_FEATS +
        [
            "speed10", "speed100", "speed100_cubed", # Added cubed
            "dir10_sin", "dir10_cos",
            "dir100_sin", "dir100_cos",
            "shear_speed", "veer_sin", "veer_cos",
            "hour_sin", "hour_cos",
            "day_sin", "day_cos"
        ] +
        [f"y_lag{L}" for L in LAGS_Y] +
        [f"y_roll{W}" for W in ROLLS_Y] +
        [f"speed10_lag{L}"  for L in LAGS_SPEED] +
        [f"speed100_lag{L}" for L in LAGS_SPEED]
    )

# ======= 2. Model Architecture (Must match Training EXACTLY) =======
class AttentionBlock(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, x):
        weights = torch.tanh(self.attn(x)) 
        weights = F.softmax(weights, dim=1)
        context = torch.sum(x * weights, dim=1) 
        return context

class AttnBiLSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout, bidirectional=True):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )
        self.rnn_out_dim = hidden_size * (2 if bidirectional else 1)
        self.attention = AttentionBlock(self.rnn_out_dim)
        self.norm = nn.LayerNorm(self.rnn_out_dim)
        self.head = nn.Sequential(
            nn.Linear(self.rnn_out_dim, self.rnn_out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.rnn_out_dim, 1),
        )

    def forward(self, x):
        o, _ = self.lstm(x)
        context = self.attention(o)
        context = self.norm(context)
        return self.head(context)

# ======= 3. Artifact Loading (Handles Ensembles) =======
def load_artifacts(model_dir: str):
    """
    Loads params, scalers, and a LIST of model states (for ensemble).
    Expects structure:
      model_dir/best_params.json
      model_dir/AttnBiLSTM/x_scaler.pkl
      model_dir/AttnBiLSTM/y_scaler.pkl
      model_dir/AttnBiLSTM/model_seed_*.pt
    """
    root = Path(model_dir)
    sub_dir = root / "AttnBiLSTM"
    
    # Load Params
    try:
        with open(root / "best_params.json", "r") as f:
            best_params = json.load(f)
    except FileNotFoundError:
        # Fallback if user put json inside the subdir
        with open(sub_dir / "best_params.json", "r") as f:
            best_params = json.load(f)
            
    # Load Scalers
    xsc = joblib.load(sub_dir / "x_scaler.pkl")
    ysc = joblib.load(sub_dir / "y_scaler.pkl")

    # Find all seed models
    model_files = list(sub_dir.glob("model_seed_*.pt"))
    if not model_files:
        raise FileNotFoundError(f"No model_seed_*.pt files found in {sub_dir}")
    
    print(f"Found {len(model_files)} models for ensemble.")
    
    states = []
    for mf in model_files:
        states.append(torch.load(mf, map_location=DEVICE))
        
    return best_params, xsc, ysc, states

def _rebuild_model(input_size: int, best_params: dict) -> AttnBiLSTMRegressor:
    model = AttnBiLSTMRegressor(
        input_size=input_size,
        hidden_size=int(best_params["hidden"]),
        num_layers=int(best_params["layers"]),
        dropout=float(best_params["dropout"]),
        bidirectional=bool(best_params["bidir"])
    ).to(DEVICE)
    return model

# ======= 4. Prediction Logic (Ensemble Aware) =======
def predict_next_from_last24(model_root: str, last24_df: pd.DataFrame, future_weather: dict|None=None):
    best, xsc, ysc, states = load_artifacts(model_root)
    
    df = last24_df.copy().sort_values(TIME_COL).reset_index(drop=True)
    lookback = int(best["lookback"])
    
    if len(df) < lookback:
        raise ValueError(f"Need at least {lookback} rows of history; got {len(df)}.")

    # Synthetic next hour row
    if future_weather is None:
        fut = df.iloc[[-1]][[TIME_COL]+BASE_FEATS].copy()
        fut[TIME_COL] = pd.to_datetime(fut[TIME_COL]) + pd.Timedelta(hours=1)
    else:
        fut = pd.DataFrame([{
            TIME_COL: pd.to_datetime(df[TIME_COL].iloc[-1]) + pd.Timedelta(hours=1),
            "U10": future_weather["U10"], "V10": future_weather["V10"],
            "U100": future_weather["U100"], "V100": future_weather["V100"],
        }])
    fut[TARGET_COL] = np.nan
    
    work = pd.concat([df[[TIME_COL, TARGET_COL]+BASE_FEATS], fut], ignore_index=True)
    dfe = add_engineered_features(work)
    
    # Feature extraction
    dfe_hist = dfe.iloc[:-1].dropna().copy()
    if len(dfe_hist) < lookback:
        raise ValueError(f"After lags, not enough rows. Provide ≥ {lookback+24} rows of history.")

    feat_cols = build_feat_list()
    X_hist = dfe_hist[feat_cols].to_numpy(np.float32)
    X_hist_s = xsc.transform(X_hist) # RobustScaler
    
    X_window = X_hist_s[-lookback:, :] # (lookback, features)
    xb = torch.from_numpy(X_window[None, ...]).float().to(DEVICE) # (1, lookback, features)
    
    # === ENSEMBLE PREDICTION ===
    preds = []
    # Rebuild architecture once
    model = _rebuild_model(X_window.shape[-1], best)
    
    for state in states:
        model.load_state_dict(state)
        model.eval()
        with torch.no_grad():
            yhat_s = model(xb).cpu().numpy()
        
        # Invert scale
        yhat = ysc.inverse_transform(yhat_s).ravel()[0]
        if bool(best["log_target"]):
            yhat = np.expm1(yhat)
        preds.append(yhat)
        
    # Average the ensemble
    final_pred = np.mean(preds)
    
    next_ts = pd.to_datetime(df[TIME_COL].iloc[-1]) + pd.Timedelta(hours=1)
    return next_ts, float(final_pred)

# ======= 5. Execution Example =======
if __name__ == "__main__":
    # --- Example Usage ---
    
    # 1. Load Data (Simulating a history file)
    # Ensure this file matches the columns used in training
    try:
        last24 = pd.read_excel("WindPowerForecastingData.xlsx") 
        # Grab just the last portion to simulate a real-time scenario
        last24 = last24.tail(50).reset_index(drop=True)
    except Exception as e:
        print(f"Could not load example data: {e}")
        exit()

    # 2. Define Root Directory (Where you ran the training script)
    # The script looks for "best_params.json" here, and "AttnBiLSTM/" subfolder
    model_root = "." 

    # 3. Optional: Define specific weather for next hour (from a weather API)
    # If None, it assumes weather persists from previous hour
    future_weather_scenario = {
        "U10": 4.5, "V10": -2.1,
        "U100": 6.8, "V100": -3.5
    }

    try:
        ts, pred_val = predict_next_from_last24(
            model_root=model_root,
            last24_df=last24,
            future_weather=future_weather_scenario 
        )
        print("\n" + "="*30)
        print(f"Prediction Time: {ts}")
        print(f"Ensemble Prediction: {pred_val:.4f} kW (or MW depending on units)")
        print("="*30 + "\n")
        
    except Exception as e:
        print(f"Prediction failed: {e}")